In [24]:
#TASK 1

In [29]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

[nltk_data] Downloading package punkt to /home/hduser/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/hduser/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/hduser/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/hduser/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [30]:
df = pd.read_csv("combined_dataset.csv")

print("Original Dataset:")
print(df.head())

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

Original Dataset:
  target                                               text
0   spam  Congratulations! You've been selected for a lu...
1   spam  URGENT: Your account has been compromised. Cli...
2   spam  You've won a free iPhone! Claim your prize by ...
3   spam  Act now and receive a 50% discount on all purc...
4   spam  Important notice: Your subscription will expir...


In [31]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [32]:
df['clean_text'] = df['text'].apply(preprocess_text)

print(df[['text', 'clean_text', 'target']].head())

df.to_csv("cleaned_emails.csv", index=False)

                                                text  \
0  Congratulations! You've been selected for a lu...   
1  URGENT: Your account has been compromised. Cli...   
2  You've won a free iPhone! Claim your prize by ...   
3  Act now and receive a 50% discount on all purc...   
4  Important notice: Your subscription will expir...   

                                          clean_text target  
0  congratulation youve selected luxury vacation ...   spam  
1  urgent account compromised click reset passwor...   spam  
2        youve free iphone claim prize clicking link   spam  
3   act receive discount purchase limited time offer   spam  
4  important notice subscription expire soon rene...   spam  


In [33]:
#TASK 2
df = pd.read_csv("cleaned_emails.csv")

print("Cleaned Dataset:")
print(df[['clean_text', 'target']].head())

Cleaned Dataset:
                                          clean_text target
0  congratulation youve selected luxury vacation ...   spam
1  urgent account compromised click reset passwor...   spam
2        youve free iphone claim prize clicking link   spam
3   act receive discount purchase limited time offer   spam
4  important notice subscription expire soon rene...   spam


In [34]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer()

# Handle potential NaN values in 'clean_text' by filling them with an empty string
df['clean_text'] = df['clean_text'].fillna('')

# Convert text into TF-IDF vectors
X = tfidf.fit_transform(df['clean_text'])
y = df['target']

# Display shape of TF-IDF matrix
print("\nTF-IDF Matrix Shape:")
print(X.shape)

# Display number of features
print("\nNumber of Features:")
print(len(tfidf.get_feature_names_out()))

# Display feature names (optional)
print("\nSample Features:")
print(tfidf.get_feature_names_out()[:20])


TF-IDF Matrix Shape:
(10961, 47419)

Number of Features:
47419

Sample Features:
['aa' 'aaa' 'aabda' 'aabvmmq' 'aac' 'aachecar' 'aaer' 'aafco' 'aah'
 'aaiabe' 'aaigrcrb' 'aaihmqv' 'aaldano' 'aalland' 'aambique' 'aamlrg'
 'aaniye' 'aaoeuro' 'aaooooright' 'aare']


In [35]:
#TASK 3
df = pd.read_csv("cleaned_emails.csv")

df['clean_text'] = df['clean_text'].fillna('')

print("Columns in dataset:", df.columns)

# Features
X_text = df['clean_text']

y = df['target']

# TF-IDF

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(X_text)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)


Columns in dataset: Index(['target', 'text', 'clean_text'], dtype='str')
Training data shape: (8768, 5000)
Testing data shape: (2193, 5000)


In [36]:
# LogisticRegression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Predictions
lr_pred = lr_model.predict(X_test)

# Accuracy
lr_accuracy = accuracy_score(y_test, lr_pred)

print("Logistic Regression Accuracy:", lr_accuracy)

# Detailed report
print("\nClassification Report:")
print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.9357045143638851

Classification Report:
              precision    recall  f1-score   support

         ham       0.93      0.99      0.96      1706
        spam       0.96      0.74      0.84       487

    accuracy                           0.94      2193
   macro avg       0.94      0.87      0.90      2193
weighted avg       0.94      0.94      0.93      2193



In [37]:
# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)

nb_accuracy = accuracy_score(y_test, nb_pred)

print("Naive Bayes Accuracy:", nb_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, nb_pred))

Naive Bayes Accuracy: 0.9430004559963521

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      0.98      0.96      1706
        spam       0.91      0.82      0.87       487

    accuracy                           0.94      2193
   macro avg       0.93      0.90      0.91      2193
weighted avg       0.94      0.94      0.94      2193



In [38]:
#TASK 4
# First split original TF-IDF data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Apply feature selection only on training data
selector = SelectKBest(chi2, k=1000)

X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

print("Original Features:", X.shape[1])
print("Reduced Features:", X_train_sel.shape[1])

Original Features: 5000
Reduced Features: 1000


In [39]:

X_train_sel, X_test_sel, y_train_sel, y_test_sel = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

In [40]:

lr_model2 = LogisticRegression(max_iter=1000)
lr_model2.fit(X_train_sel, y_train_sel)

pred_sel = lr_model2.predict(X_test_sel)

reduced_accuracy = accuracy_score(y_test_sel, pred_sel)

print("Accuracy After Feature Selection:", reduced_accuracy)

Accuracy After Feature Selection: 0.925672594619243


In [41]:
#TASK 5
print("Accuracy Before Reduction:", lr_accuracy)
print("Accuracy After Reduction:", reduced_accuracy)

Accuracy Before Reduction: 0.9357045143638851
Accuracy After Reduction: 0.925672594619243
